Wavelet Block Thresholding
==========================

*Important:* Please read the [installation page](http://gpeyre.github.io/numerical-tours/installation_python/) for details about how to install the toolboxes.
$\newcommand{\dotp}[2]{\langle #1, #2 \rangle}$
$\newcommand{\enscond}[2]{\lbrace #1, #2 \rbrace}$
$\newcommand{\pd}[2]{ \frac{ \partial #1}{\partial #2} }$
$\newcommand{\umin}[1]{\underset{#1}{\min}\;}$
$\newcommand{\umax}[1]{\underset{#1}{\max}\;}$
$\newcommand{\umin}[1]{\underset{#1}{\min}\;}$
$\newcommand{\uargmin}[1]{\underset{#1}{argmin}\;}$
$\newcommand{\norm}[1]{\|#1\|}$
$\newcommand{\abs}[1]{\left|#1\right|}$
$\newcommand{\choice}[1]{ \left\{  \begin{array}{l} #1 \end{array} \right. }$
$\newcommand{\pa}[1]{\left(#1\right)}$
$\newcommand{\diag}[1]{{diag}\left( #1 \right)}$
$\newcommand{\qandq}{\quad\text{and}\quad}$
$\newcommand{\qwhereq}{\quad\text{where}\quad}$
$\newcommand{\qifq}{ \quad \text{if} \quad }$
$\newcommand{\qarrq}{ \quad \Longrightarrow \quad }$
$\newcommand{\ZZ}{\mathbb{Z}}$
$\newcommand{\CC}{\mathbb{C}}$
$\newcommand{\RR}{\mathbb{R}}$
$\newcommand{\EE}{\mathbb{E}}$
$\newcommand{\Zz}{\mathcal{Z}}$
$\newcommand{\Ww}{\mathcal{W}}$
$\newcommand{\Vv}{\mathcal{V}}$
$\newcommand{\Nn}{\mathcal{N}}$
$\newcommand{\NN}{\mathcal{N}}$
$\newcommand{\Hh}{\mathcal{H}}$
$\newcommand{\Bb}{\mathcal{B}}$
$\newcommand{\Ee}{\mathcal{E}}$
$\newcommand{\Cc}{\mathcal{C}}$
$\newcommand{\Gg}{\mathcal{G}}$
$\newcommand{\Ss}{\mathcal{S}}$
$\newcommand{\Pp}{\mathcal{P}}$
$\newcommand{\Ff}{\mathcal{F}}$
$\newcommand{\Xx}{\mathcal{X}}$
$\newcommand{\Mm}{\mathcal{M}}$
$\newcommand{\Ii}{\mathcal{I}}$
$\newcommand{\Dd}{\mathcal{D}}$
$\newcommand{\Ll}{\mathcal{L}}$
$\newcommand{\Tt}{\mathcal{T}}$
$\newcommand{\si}{\sigma}$
$\newcommand{\al}{\alpha}$
$\newcommand{\la}{\lambda}$
$\newcommand{\ga}{\gamma}$
$\newcommand{\Ga}{\Gamma}$
$\newcommand{\La}{\Lambda}$
$\newcommand{\si}{\sigma}$
$\newcommand{\Si}{\Sigma}$
$\newcommand{\be}{\beta}$
$\newcommand{\de}{\delta}$
$\newcommand{\De}{\Delta}$
$\newcommand{\phi}{\varphi}$
$\newcommand{\th}{\theta}$
$\newcommand{\om}{\omega}$
$\newcommand{\Om}{\Omega}$

This numerical tour presents block thresholding methods,
that makes use of the structure of wavelet coefficients of natural images to perform denoising.
Theoretical properties of block thresholding were investigated
in [CaiSilv](#biblio) [Cai99](#biblio) [HallKerkPic99](#biblio)

In [ ]:
from __future__ import division
import nt_toolbox as nt
from nt_solutions import denoisingwav_4_block as solutions
%matplotlib inline
%load_ext autoreload
%autoreload 2

Generating a Noisy Image
------------------------
Here we use an additive Gaussian noise.


Size of the image of $N=n \times n$ pixels.

In [ ]:
n = 256

First we load an image $f_0 \in \RR^N$.

In [ ]:
name = 'boat'
f0 = rescale(load_image(name, n))

Display it.

In [ ]:
clf; imageplot(f0)

Noise level.

In [ ]:
sigma = .08

Generate a noisy image $f=f_0+\epsilon$ where $\epsilon \sim
\Nn(0,\si^2\text{Id}_N)$.

In [ ]:
f = f0 + sigma*randn(size(f0))

Display it.

In [ ]:
clf; imageplot(clamp(f))

Orthogonal Wavelet Thresholding
-------------------------------
We first consider the traditional
wavelet thresholding method.


Parameters for the orthogonal wavelet transform.

In [ ]:
Jmin = 4
options.ti = 0

Shortcuts for the foward and backward wavelet transforms.

In [ ]:
wav  = lambda f: perform_wavelet_transf(f, Jmin, + 1, options)
iwav = lambda fw: perform_wavelet_transf(fw, Jmin, -1, options)

Display the original set of noisy coefficients.

In [ ]:
plot_wavelet(wav(f), Jmin)

Denoting $\Ww$ and $\Ww^*$ the forward and backward wavelet
transform, wavelet thresholding $\tilde f$ is defined as
$$ \tilde f = \Ww^* \circ \theta_T \circ \Ww(f) $$
where $T>0$ is the threshold, that should be adapted to the noise
level.


The thresholding operator is applied component-wise
$$ \th_T(x)_i = \psi_T(x_i) x_i $$
where $\psi_T$ is an atenuation fonction. In this tour, we use the James
Stein (JS) attenuation:
$$ \psi_T(s) = \max\pa{ 0, 1-\frac{T^2}{s^2} } $$

In [ ]:
psi = lambda s, T: max3(1-T^2 ./ max(abs(s).^2, 1e-9), 0)

Display the thresholding function $\th_T$.

In [ ]:
t = linspace(-3, 3, 1024)
clf; hold on
plot(t, t.*psi(t, 1))
plot(t, t, 'r--'); axis equal

Thresholding operator.

In [ ]:
theta = lambda x, T: psi(x, T).*x
ThreshWav = lambda f, T: iwav(theta(wav(f), T))

Test the thresholding.

In [ ]:
T = 1.5*sigma

imageplot(clamp(ThreshWav(f, T)))

__Exercise 1__

Display the evolution of the denoising SNR when $T$ varies.
Store in |fThresh| the optimal denoising result.
etrieve best

In [ ]:
solutions.exo1()

In [ ]:
## Insert your code here.

Display the optimal thresolding.

In [ ]:
imageplot(clamp(fThresh), strcat(['SNR = ' num2str(snr(f0, fThresh), 3)]))

Block Thresholding Operator
---------------------------
A block thresholding operator of coefficients $x=(x_i)_{i=1}^P \in \RR^P$ is defined
using a dijoint partition into a set of blocks $B$
$$ \{1,\ldots,P\} = \bigcup_{b \in B} b. $$
Its definition reads
$$ \forall i \in b, \quad
      \theta_T(x)_i = \psi_T( \norm{x_b}_2 ) x_i $$
where $ x_b = (x_j)_{j \in B} \in \RR^{\abs{b}} $.
One thus thresholds the $\ell^2$ norm (the energy) of each block rather
than each coefficient independently.


For image-based thresholding, we use a partition in square blocks of
equal size $w \times w$.


The block size $w$.

In [ ]:
w = 4

Compute indexing of the blocks.

In [ ]:
[dX, dY, X, Y] = ndgrid(0: w-1, 0: w-1, 1: w: n-w + 1, 1: w: n-w + 1)
I = X + dX + (Y + dY-1)*n

Block extraction operator. It returns the set of $ \{x_b\}_{b \in B} $
of block-partitioned coefficients.

In [ ]:
block = lambda x: reshape(x(I(: )), size(I))

Block reconstruction operator.

In [ ]:
iblock = lambda H: assign(zeros(n), I, H)

Check that block extraction / reconstruction gives perfect
reconstruction.

In [ ]:
mynorm = lambda x: norm(x(: ))
fprintf('Should be 0: %.3f\n', mynorm(f - iblock(block(f))))

Compute the average energy of each block, and duplicate.

In [ ]:
repm = lambda v: repmat(max3(v, 1e-15), [w w])
energy = lambda H: repm(sqrt(mean(mean(abs(H).^2, 1), 2)))

Block thresholding operator.

In [ ]:
Thresh = lambda H, T: psi(energy(H), T).*H
ThreshBlock = lambda x, T: iblock(Thresh(block(x), T))

__Exercise 2__

Test the effect of block thresholding on the image $f_0$ itself, for increasing value of $T$.
(of course thresholding directly the image has no interest, this is just
to vizualize the effect).

In [ ]:
solutions.exo2()

In [ ]:
## Insert your code here.

Orthogonal Wavelet Block Thresholding
-------------------------------------
Wavelet coefficients of natural images are not independant one from each
other. One can thus improve the denoising results by thresholding block
of coefficients togethers. Block thresholding is only efficient when
used as a soft thresholder. Here we use a Stein soft thresholder.


Display the thresholded coefficients for a threshold value $T$ proportional to the noise level $\si$.

In [ ]:
T = 1.25*sigma

plot_wavelet(ThreshBlock(wav(f), T), Jmin)

Define the wavelet block thresholding operator.

In [ ]:
ThreshWav = lambda f, T: iwav(ThreshBlock(wav(f), T))

Test the thresholding.

In [ ]:
imageplot(clamp(ThreshWav(f, T)))

__Exercise 3__

Display the evolution of the denoising SNR when $T$ varies.
Store in |fBlock| the optimal denoising result.
etrieve best

In [ ]:
solutions.exo3()

In [ ]:
## Insert your code here.

Display the result.

In [ ]:
imageplot(clamp(fBlock), strcat(['SNR = ' num2str(snr(f0, fBlock), 3)]))

Translation invariant Block Thresholding
----------------------------------------
Block thresholding can also be applied to a translation invariant wavelet
transform. It gives state of the art denoising results.



Shortcuts for the foward and backward translation invariant wavelet transforms.

In [ ]:
options.ti = 1
wav  = lambda f: perform_wavelet_transf(f, Jmin, + 1, options)
iwav = lambda fw: perform_wavelet_transf(fw, Jmin, -1, options)

Foward wavelet transform.

In [ ]:
fw = wav(f)

Compute indexing of the blocks.

In [ ]:
[dX, dY, X, Y, J] = ndgrid(0: w-1, 0: w-1, 1: w: n-w + 1, 1: w: n-w + 1, 1: size(fw, 3))
I = X + dX + (Y + dY-1)*n + (J-1)*n^2

Forward and backward extraction operators.

In [ ]:
block = lambda x: reshape(x(I(: )), size(I))
iblock = lambda H: assign(zeros(size(fw)), I, H)

Compute the average energy of each block, and duplicate.

In [ ]:
repm = lambda v: repmat(max3(v, 1e-15), [w w])
energy = lambda H: repm(sqrt(mean(mean(abs(H).^2, 1), 2)))

Block thresholding operator.

In [ ]:
Thresh = lambda H, T: psi(energy(H), T).*H
ThreshBlock = lambda x, T: iblock(Thresh(block(x), T))

Define the wavelet block thresholding operator.

In [ ]:
ThreshWav = lambda f, T: iwav(ThreshBlock(wav(f), T))

Test the thresholding.

In [ ]:
T = 1.25*sigma
imageplot(clamp(ThreshWav(f, T)))

__Exercise 4__

Display the evolution of the denoising SNR when $T$ varies.
Store in |fTI| the optimal denoising result.
etrieve best

In [ ]:
solutions.exo4()

In [ ]:
## Insert your code here.

Display the result.

In [ ]:
imageplot(clamp(fTI), strcat(['SNR = ' num2str(snr(f0, fTI), 3)]))

Bibliography
------------
<html><a name="biblio"></a></html>


* [CaiSil01] T. Cai and B.W. Silverman, [Incorporating information on neighboring coefficients into wavelet estimation][1], Sankhya 63, 127-148, 2001.
* [Cai99] T. Cai, [Adaptive wavelet estimation: a block thresholding and oracle inequality approach][2], The Annals of Statistics 27, 898-924, 1999.
* [HallKerkPic99] P. Hall, G. Kerkyacharian and D. Picard, _On the minimax optimality of block thresholded wavelet estimator_, Statistica Sinica 9(1999), 33-49

[1]:http://sankhya.isical.ac.in/search/63b2/caifnl.html
[2]:http://dx.doi.org/10.1214/aos/1018031262